In [ ]:
%reset -f

# Surface DAS Forward Modeling

Acoustic 2-D forward modeling for a **Surface DAS** acquisition geometry.

Key differences from the Borehole DAS-VSP workflow:
| | Borehole DAS-VSP | Surface DAS |
|---|---|---|
| Receiver array | Vertical – constant x, varying z | Horizontal – varying x, constant z ≈ 1 m |
| Source | Single surface shot (fixed offset) | Multiple surface shots (roll-along) |
| Coupling | Tube wave / guided wave | Direct/reflected surface waves + body waves |
| DAS response | Axial strain along borehole | Axial strain along surface cable (≈ ∂vₓ/∂x) |

This notebook demonstrates:
1. Building a 2-D velocity model from the sonic log  
2. Defining the surface DAS geometry  
3. Running acoustic forward modeling with **pyseis**  
4. Visualising shot gathers  
5. Computing the DAS strain-rate proxy via spatial differentiation  
6. Saving the synthetic data

In [ ]:
import plotting
from pyseis.wave_equations import acoustic_isotropic
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage

import os
pwd = os.getcwd()
%env DATAPATH={pwd}/wrk/

## 1  Model Grid and Velocity Model

In [ ]:
# ---- model grid ----
ox, oz = 0.0, 0.0
dx, dz = 1.0, 1.0      # grid spacing (m)
x1, x2 = 0.0, 550.0
z1, z2 = 0.0, 900.0

x_grid = np.arange(x1, x2 + dx, dx)
z_grid = np.arange(z1, z2 + dz, dz)
nx, nz = x_grid.size, z_grid.size
print(f"Grid dimensions : nx={nx}, nz={nz}")

model_sampling = (dx, dz)
model_origins  = (ox, oz)
model_padding  = (100, 100)

In [ ]:
# ---- build 2-D velocity model from sonic log ----
df   = pd.read_csv('Curtin_DAS_VSP_Sonic_modified.csv')
log_depth = df['Depth (DAS)'].values
log_vel   = df['interval velocity'].values

# interpolate 1-D profile onto the model z-grid
vel_1d = np.interp(z_grid, log_depth, log_vel,
                   left=log_vel[0], right=log_vel[-1]).astype(np.float32)

# replicate laterally → horizontally homogeneous 2-D model  shape (nx, nz)
vp2d = np.tile(vel_1d, (nx, 1))


def _clean_velocity(v, v_floor=1500.0):
    """Fill NaNs by vertical interpolation; clip to v_floor."""
    v = np.array(v, copy=True)
    n_x, n_z = v.shape
    zidx = np.arange(n_z)
    for ix in range(n_x):
        col = v[ix, :]
        m = np.isfinite(col)
        if m.any():
            v[ix, :] = np.interp(zidx, zidx[m], col[m])
        else:
            j = min(range(n_x), key=lambda k: (abs(k - ix), not np.isfinite(v[k, :]).any()))
            v[ix, :] = v[j, :]
    return np.clip(v, v_floor, None).astype(np.float32)


vp2d = _clean_velocity(vp2d)
print(f"Velocity model shape : {vp2d.shape}")
print(f"Vp range             : {vp2d.min():.0f} – {vp2d.max():.0f} m/s")

## 2  Surface DAS Acquisition Geometry

In [ ]:
# ---- Surface DAS receiver cable (horizontal, at z = 1 m) ----
drec      = 5.0    # receiver spacing along cable (m)
recx1     = 50.0   # cable start (m)
recx2     = 500.0  # cable end   (m)
recz_surf = 1.0    # depth of surface cable (m)

recx = np.arange(recx1, recx2 + drec, drec)
recz = np.full(recx.shape, recz_surf)
recs = np.column_stack([recx, recz])

# ---- Surface shot points (roll-along, also at z = 1 m) ----
# Shots are placed outside and within the receiver spread
srcx = np.array([25.0, 137.5, 250.0, 362.5, 475.0])
srcz = np.full(srcx.shape, 1.0)
srcs = np.column_stack([srcx, srcz])

print(f"Receiver cable  : {len(recx)} channels,  "
      f"x = {recx[0]:.0f}–{recx[-1]:.0f} m,  spacing = {drec:.0f} m")
print(f"Shot positions  : {len(srcx)} shots,  x = {srcx}")
print(f"recs.shape      : {recs.shape}")
print(f"srcs.shape      : {srcs.shape}")

In [ ]:
plotting.plot_model(
    vp2d,
    title='Velocity Model with Surface DAS Geometry',
    cbar=True,
    sampling=model_sampling,
    origins=model_origins,
    cmap='jet',
    cbar_label='$v_p$ (m/s)',
    src_locations=srcs,
    rec_locations=recs,
    figsize=(12, 6),
    vlims=[1400, 2500],
)

## 3  Source Wavelet

A **Ricker wavelet** (40 Hz dominant frequency) is used as the surface source.  
Alternatively, load a field-estimated source from `../FWI/source_pilot_run3_raw.bin`.

In [ ]:
# ---- Ricker wavelet ----
dt = 0.001    # time step (s)
nt = 2000     # total time samples  →  T_max = 2.0 s
f0 = 40.0     # dominant frequency (Hz)

t   = np.arange(nt) * dt
t0  = 1.0 / f0                                      # wavelet peak time (s)
tau = t - t0
source_form = ((1.0 - 2.0 * (np.pi * f0 * tau)**2) *
               np.exp(-(np.pi * f0 * tau)**2)).astype(np.float32)

# ---- display ----
freqs = np.fft.rfftfreq(nt, dt)
spec  = np.abs(np.fft.rfft(source_form))

fig, axs = plt.subplots(1, 2, figsize=(11, 3))
axs[0].plot(t, source_form); axs[0].set_xlabel('Time (s)')
axs[0].set_ylabel('Amplitude'); axs[0].set_title(f'Ricker wavelet  f₀ = {f0} Hz')
axs[0].grid(True)
axs[1].plot(freqs, spec); axs[1].set_xlim(0, 150)
axs[1].set_xlabel('Frequency (Hz)'); axs[1].set_ylabel('|X(f)|')
axs[1].set_title('Amplitude Spectrum'); axs[1].grid(True)
plt.tight_layout(); plt.show()

## 4  Acoustic Forward Modeling

In [ ]:
# ---- build acoustic 2-D solver ----
acoustic_surface_das = acoustic_isotropic.AcousticIsotropic2D(
    model=vp2d,
    model_sampling=model_sampling,
    model_padding=model_padding,
    wavelet=source_form,
    d_t=dt,
    src_locations=srcs,
    rec_locations=recs,
    model_origins=model_origins,
    subsampling=20,
    gpus=[0])

In [ ]:
%%time
data_surface_das = acoustic_surface_das.forward(vp2d)
# Output shape: (n_shots, n_recs, nt)
print(f"Synthetic data shape : {data_surface_das.shape}")
print(f"  axis-0  →  shots  ({data_surface_das.shape[0]})")
print(f"  axis-1  →  receivers  ({data_surface_das.shape[1]})")
print(f"  axis-2  →  time samples  ({data_surface_das.shape[2]})")

## 5  Shot Gather Visualisation

Plot up to 3 representative shot gathers to inspect moveout and reflections.

In [ ]:
plotting.plot_data(
    data_surface_das,
    d_t=dt,
    src_locations=srcx,
    rec_locations=recx,
    n_shots=min(3, len(srcx)),
    figsize=(5, 7),
    pclip=95,
    cmap='gray',
    xlabel='Receiver x (m)',
    ylabel='Time (s)',
)

In [ ]:
# ---- detailed view: one shot gather per row ----
n_show = min(len(srcx), 3)
fig, axes = plt.subplots(1, n_show, figsize=(5 * n_show, 7), sharey=True)
if n_show == 1:
    axes = [axes]

clip = np.percentile(np.abs(data_surface_das), 95)
t_axis = np.arange(nt) * dt

for k, ax in enumerate(axes):
    shot_idx = round(k * (len(srcx) - 1) / max(n_show - 1, 1))
    ax.pcolormesh(recx, t_axis, data_surface_das[shot_idx].T,
                  shading='nearest', cmap='seismic',
                  vmin=-clip, vmax=clip)
    ax.invert_yaxis()
    ax.set_title(f'Shot x = {srcx[shot_idx]:.1f} m')
    ax.set_xlabel('Receiver x (m)')
    if k == 0:
        ax.set_ylabel('Time (s)')

plt.suptitle('Surface DAS – Synthetic Shot Gathers', fontsize=13)
plt.tight_layout()
plt.show()

## 6  DAS Strain-Rate Proxy

For a **horizontal** DAS cable, the fibre measures the **longitudinal strain rate**:

$$\dot{\varepsilon}_{xx}(x,t) = \frac{\partial v_x}{\partial x}$$

Because the acoustic (pressure) solver does not directly output particle velocity, we use the
following first-order approximation to convert the pressure gather `P(x,t)` to a
DAS-equivalent trace:

$$\text{DAS}(x,t) \approx \frac{\partial}{\partial x}\left(\frac{\partial P}{\partial t}\right)$$

Implemented as successive finite differences along the receiver and time axes.

In [ ]:
# ---- DAS strain-rate proxy: ∂²P / ∂x ∂t ----
# time derivative along axis-2, then spatial derivative along axis-1
dP_dt   = np.gradient(data_surface_das, dt,   axis=2)   # shape unchanged
das_data = np.gradient(dP_dt,            drec, axis=1)

# ---- plot DAS gathers ----
n_show = min(len(srcx), 3)
fig, axes = plt.subplots(1, n_show, figsize=(5 * n_show, 7), sharey=True)
if n_show == 1:
    axes = [axes]

clip_das = np.percentile(np.abs(das_data), 95)

for k, ax in enumerate(axes):
    shot_idx = round(k * (len(srcx) - 1) / max(n_show - 1, 1))
    ax.pcolormesh(recx, t_axis, das_data[shot_idx].T,
                  shading='nearest', cmap='seismic',
                  vmin=-clip_das, vmax=clip_das)
    ax.invert_yaxis()
    ax.set_title(f'DAS  shot x = {srcx[shot_idx]:.1f} m')
    ax.set_xlabel('Receiver x (m)')
    if k == 0:
        ax.set_ylabel('Time (s)')

plt.suptitle('Surface DAS – Strain-Rate Proxy (∂²P/∂x∂t)', fontsize=13)
plt.tight_layout()
plt.show()

## 7  Save Synthetic Data

In [ ]:
import os

os.makedirs('./data', exist_ok=True)

# save pressure gathers  (n_shots, n_recs, nt)  float32
out_pressure = './data/Surface_DAS_pressure.bin'
data_surface_das.astype(np.float32).tofile(out_pressure)
print(f"Saved pressure gathers → {out_pressure}  shape {data_surface_das.shape}")

# save DAS strain-rate proxy
out_das = './data/Surface_DAS_strainrate.bin'
das_data.astype(np.float32).tofile(out_das)
print(f"Saved DAS strain-rate  → {out_das}  shape {das_data.shape}")

# metadata for downstream workflows
meta = {
    'nx': int(nx), 'nz': int(nz),
    'dx': dx, 'dz': dz,
    'ox': ox, 'oz': oz,
    'dt': dt, 'nt': int(nt),
    'f0': f0,
    'n_shots': int(len(srcx)),
    'n_recs':  int(len(recx)),
    'drec': drec,
    'recx1': recx1, 'recx2': recx2, 'recz_surf': recz_surf,
    'srcx': srcx.tolist(),
}
import json
with open('./data/Surface_DAS_meta.json', 'w') as fh:
    json.dump(meta, fh, indent=2)
print("Saved metadata          → ./data/Surface_DAS_meta.json")